In [1]:
from pathlib import Path
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from dotenv import load_dotenv

env_path = Path.cwd() / ".env"
if not env_path.exists():
    env_path = Path.cwd().parent / ".env"
load_dotenv(env_path)  # Load environment variables from workspace root .env file

llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
)


In [2]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# ── State ─────────────────────────────────────────────────────────────────────
class State(TypedDict):
    topic: str
    content: str
    critique: str
    iteration: int
    max_iterations: int

# ── Node 1: Generate / Refine content ────────────────────────────────────────
def generate(state: State) -> dict:
    iteration = state["iteration"] + 1
    print(f"[generate] iteration {iteration}")

    if state["content"] == "":
        # First pass — write from scratch
        prompt = f"Write a concise 3-sentence paragraph about: {state['topic']}"
    else:
        # Subsequent passes — refine based on critique
        prompt = (
            f"Rewrite and improve the following paragraph based on the critique.\n\n"
            f"Paragraph:\n{state['content']}\n\n"
            f"Critique:\n{state['critique']}\n\n"
            f"Return only the improved paragraph."
        )

    response = llm.invoke([
        SystemMessage(content="You are a skilled writer who produces clear, engaging content."),
        HumanMessage(content=prompt),
    ])
    return {"content": response.content, "iteration": iteration}

# ── Node 2: Critique content ──────────────────────────────────────────────────
def critique(state: State) -> dict:
    print(f"[critique] iteration {state['iteration']}")
    response = llm.invoke([
        SystemMessage(content=(
            "You are a strict editor. Review the paragraph and provide 1-2 specific, "
            "actionable improvements. Be concise."
        )),
        HumanMessage(content=f"Paragraph:\n{state['content']}"),
    ])
    return {"critique": response.content}

# ── Conditional edge: loop or stop ────────────────────────────────────────────
def should_continue(state: State) -> str:
    if state["iteration"] < state["max_iterations"]:
        print(f"[router] iteration {state['iteration']} < {state['max_iterations']} → looping back")
        return "generate"
    print(f"[router] max iterations reached → END")
    return END

# ── Build Graph ───────────────────────────────────────────────────────────────
builder = StateGraph(State)

builder.add_node("generate", generate)
builder.add_node("critique", critique)

# START → generate
builder.add_edge(START, "generate")

# generate → critique
builder.add_edge("generate", "critique")

# critique → conditional: loop back to generate OR finish
builder.add_conditional_edges("critique", should_continue, ["generate", END])

graph = builder.compile()
print("Iterative graph compiled successfully.")


Iterative graph compiled successfully.


In [3]:
# ── Run the iterative graph ───────────────────────────────────────────────────
initial_state = State(
    topic="The importance of sleep for cognitive performance",
    content="",
    critique="",
    iteration=0,
    max_iterations=3,
)

result = graph.invoke(initial_state)

print("\n" + "=" * 60)
print(f"TOPIC : {initial_state['topic']}")
print(f"TOTAL ITERATIONS : {result['iteration']}")
print("=" * 60)
print("\n--- FINAL CONTENT (after iterative refinement) ---")
print(result["content"])
print("\n--- LAST CRITIQUE ---")
print(result["critique"])


[generate] iteration 1
[critique] iteration 1
[router] iteration 1 < 3 → looping back
[generate] iteration 2
[critique] iteration 2
[router] iteration 2 < 3 → looping back
[generate] iteration 3
[critique] iteration 3
[router] max iterations reached → END

TOPIC : The importance of sleep for cognitive performance
TOTAL ITERATIONS : 3

--- FINAL CONTENT (after iterative refinement) ---
Sleep is vital for cognitive performance. It facilitates memory consolidation, enhances learning, and supports problem-solving and adaptability to new challenges. During sleep, the brain processes and organizes information, strengthening neural connections essential for retaining knowledge and cognitive flexibility. In contrast, chronic sleep deprivation impairs attention, decision-making, and creativity, underscoring the necessity of quality sleep for optimal mental function.

--- LAST CRITIQUE ---
1. Replace "supports problem-solving and adaptability to new challenges" with "enhances problem-solving and